# PurffleShorts in Google Colab: free AI YouTube Shorts generator

Make finished vertical videos for YouTube Shorts, TikTok and Instagram Reels in your browser, with nothing to
install on your computer. Give it a topic, a web page, or a long video to clip. It writes the script (any LLM),
voices it with free neural voices, adds footage, word-synced captions, music and sound effects, and renders
with ffmpeg.

**Run the cells from top to bottom.** The first demo needs no keys at all.

- Clipping long videos is faster on a GPU: *Runtime → Change runtime type → T4 GPU* (optional).
- Keys are read from Colab **Secrets** (the key icon on the left), never typed into the notebook.
- Source, docs and issues: [github.com/Chamanrajragu/purffle-shorts](https://github.com/Chamanrajragu/purffle-shorts) (a ⭐ helps other creators find it).

In [ ]:
#@title 1. Install PurffleShorts (about a minute)
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab already has ffmpeg and OpenCV (face framing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "purffle-shorts[clip] @ git+https://github.com/Chamanrajragu/purffle-shorts"], check=True)
os.makedirs("/content/purffle" if IN_COLAB else "purffle-colab", exist_ok=True)
os.chdir("/content/purffle" if IN_COLAB else "purffle-colab")
os.environ.setdefault("UPLOAD", "false")   # nothing is uploaded from this notebook
subprocess.run([sys.executable, "-m", "purffle_shorts", "--version"], check=True)

In [ ]:
#@title 2. A demo Short with no keys (sample script, free voice, generated backgrounds)
import glob

from IPython.display import Video, display

STYLE = "chat"  #@param ["facts", "chat", "dialogue", "reddit"]


def latest_video():
    files = sorted(glob.glob("output_videos/*/*.mp4"), key=os.path.getmtime)
    return files[-1] if files else None


subprocess.run([sys.executable, "-m", "purffle_shorts", "demo", "--style", STYLE], check=True)
display(Video(latest_video(), embed=True, width=320))

## Use your own AI model and footage

Add these in **Secrets** (key icon on the left) and switch on *Notebook access* for each:

| Secret | Where to get it | What for |
|---|---|---|
| `GEMINI_API_KEY` | [aistudio.google.com](https://aistudio.google.com/apikey) (free tier) | writes the script |
| or `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY` | | any one LLM is enough |
| `PEXELS_API_KEY` | [pexels.com/api](https://www.pexels.com/api/) (free) | stock footage per scene |
| `PIXABAY_API_KEY` | [pixabay.com/api/docs](https://pixabay.com/api/docs/) (free) | more footage |

In [ ]:
#@title 3. Load your keys from Colab Secrets
KEYS = ["GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY",
        "DEEPSEEK_API_KEY", "PEXELS_API_KEY", "PIXABAY_API_KEY", "ELEVENLABS_API_KEY"]
try:
    from google.colab import userdata
    for k in KEYS:
        try:
            value = userdata.get(k)
        except Exception:
            value = None
        if value:
            os.environ[k] = value
            print("✔", k)
except ImportError:
    print("Not in Colab: using this machine's environment and .env")
subprocess.run([sys.executable, "-m", "purffle_shorts", "doctor"])

In [ ]:
#@title 4. Make a Short about anything
TOPIC = "why octopuses have three hearts"  #@param {type:"string"}
FORMAT = "auto"  #@param ["auto", "facts", "story", "listicle", "myth", "quiz", "explainer", "dialogue", "chat", "reddit"]
LANGUAGE = "en"  #@param {type:"string"}
SECONDS = 40  #@param {type:"slider", min:15, max:90, step:5}
CAPTIONS = "bold"  #@param ["bold", "boxed", "neon", "clean", "karaoke", "minimal"]
subprocess.run([sys.executable, "-m", "purffle_shorts", "make", "--topic", TOPIC, "--style", FORMAT,
                "--lang", LANGUAGE, "--duration", str(SECONDS), "--caption-style", CAPTIONS, "--no-upload"],
               check=True)
display(Video(latest_video(), embed=True, width=320))

In [ ]:
#@title 5. Clip a long video (podcast, interview, talk) into Shorts
#@markdown Upload a video with the folder icon on the left (or give a URL of a video you may reuse).
SOURCE = "/content/podcast.mp4"  #@param {type:"string"}
CLIPS = 3  #@param {type:"slider", min:1, max:8, step:1}
FRAMING = "face"  #@param ["face", "center", "blur"]
BRIEF = ""  #@param {type:"string"}
#@markdown BRIEF is optional, e.g. "the funniest moments" (needs an AI key).
cmd = [sys.executable, "-m", "purffle_shorts", "clip", SOURCE, "--count", str(CLIPS), "--crop", FRAMING,
       "--no-upload"] + (["--prompt", BRIEF] if BRIEF.strip() else [])
subprocess.run(cmd, check=True)
for f in sorted(glob.glob("output_videos/*/*.mp4"), key=os.path.getmtime)[-CLIPS:]:
    print(f)
    display(Video(f, embed=True, width=280))

In [ ]:
#@title 6. Download everything you made
import shutil

archive = shutil.make_archive("purffle-videos", "zip", "output_videos")
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    print(archive)

**Next steps**

- Upload to YouTube automatically, schedule posts and run it every day on your own computer or a server:
  see the [README](https://github.com/Chamanrajragu/purffle-shorts#-quick-start).
- The Studio web app, the MCP server for Claude and Cursor, and the REST API for n8n run locally.